Using an ensemble of models with architecture 784,784,784,10 to find the exemplary digits turned out to be a bad idea. The result seemed random but it was the same random distributio for all of them, implying they learned the same things through different minima.

This notebook is tot construct models with different architecture in the ensemble, this might lead to differing exemplary digits for each which would lead to a conbined result that looks like a valid digit.

In [ ]:
from torch import nn
from torchvision import datasets
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch import no_grad
from torch import argmax
from torchvision import transforms
from torch import cuda
from torch import save

In [ ]:
# creating the model class
class Ann(nn.Module):
  def __init__(self, num_layers, hidden_units):
    assert num_layers >= 2, "Number of layers must be greater than or equal to 2"
    super().__init__()
    self.first = nn.Linear(784, hidden_units)
    self.hidden_layers = nn.ModuleList()

    for i in range(num_layers - 2):
      self.hidden_layers.append(nn.Linear(hidden_units, hidden_units))

    self.last = nn.Linear(hidden_units, 10)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.relu(self.first(x))
    for layer in self.hidden_layers:
      x = self.relu(layer(x))
    x = self.last(x)
    return x

In [ ]:
# initialize the dataloader for mnist
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 501kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.98MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.86MB/s]


In [ ]:
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
# hyperparameters
lr = 0.001
epochs = 10

In [ ]:
# ensemble with mixed fully connected architectures
judges = [None]*5
judges[0] = Ann(3, 784)
judges[1] = Ann(2, 1000)
judges[2] = Ann(5, 300)
judges[3] = Ann(2, 250)
judges[4] = Ann(4, 800)

In [ ]:
for network in judges:
  network.to(device)

In [ ]:
# train all of them
for i, network in enumerate(judges):
  optimizer = Adam(network.parameters(), lr=lr)
  criterion = nn.CrossEntropyLoss()

  for epoch in range(epochs):
    for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)
      x = images.view(images.shape[0], -1)
      output = network(x)
      loss = criterion(output, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    # calculate accuracy on validation set
    total = 0
    correct = 0
    with no_grad():
      for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        x = images.view(images.shape[0], -1)
        output = network(x)
        predictions = argmax(output, dim=1)
        total += labels.shape[0]
        correct += (predictions == labels).sum().item()
      print(f'Judge number: {i+1}. Epoch: {epoch+1}. Train Loss: {loss.item()}. Test Accuracy: {100*correct/total:.2f}')

Judge number: 1. Epoch: 1. Train Loss: 0.03764752298593521. Test Accuracy: 96.58
Judge number: 1. Epoch: 2. Train Loss: 0.025947291404008865. Test Accuracy: 97.43
Judge number: 1. Epoch: 3. Train Loss: 0.07189575582742691. Test Accuracy: 97.92
Judge number: 1. Epoch: 4. Train Loss: 0.026060810312628746. Test Accuracy: 98.17
Judge number: 1. Epoch: 5. Train Loss: 0.03109741397202015. Test Accuracy: 98.00
Judge number: 1. Epoch: 6. Train Loss: 0.01107339933514595. Test Accuracy: 97.92
Judge number: 1. Epoch: 7. Train Loss: 0.029288966208696365. Test Accuracy: 97.81
Judge number: 1. Epoch: 8. Train Loss: 0.002150552347302437. Test Accuracy: 97.98
Judge number: 1. Epoch: 9. Train Loss: 0.029420696198940277. Test Accuracy: 97.94
Judge number: 1. Epoch: 10. Train Loss: 0.0004510876315180212. Test Accuracy: 97.91
Judge number: 2. Epoch: 1. Train Loss: 0.2810055613517761. Test Accuracy: 96.51
Judge number: 2. Epoch: 2. Train Loss: 0.23782461881637573. Test Accuracy: 97.57
Judge number: 2. Epoc

In [ ]:
# create folder ensemble
!mkdir mixed_ensemble
# saving the models
for i, network in enumerate(judges):
  save(network.state_dict(), f'mixed_ensemble/judge_{i+1}.pth')

In [ ]:
# zip it
!zip -r mixed_ensemble.zip mixed_ensemble

  adding: mixed_ensemble/ (stored 0%)
  adding: mixed_ensemble/judge_3.pth (deflated 7%)
  adding: mixed_ensemble/judge_1.pth (deflated 7%)
  adding: mixed_ensemble/judge_4.pth (deflated 7%)
  adding: mixed_ensemble/judge_2.pth (deflated 7%)
  adding: mixed_ensemble/judge_5.pth (deflated 7%)
